# 07 – The quoted cost of a stay of about one month
## Question
How do 30-night equivalent quoted costs compare between Barcelona and Madrid, for entire homes and private rooms?
Use recorded check-in/out dates and `price_quote_total_price` for stays of 28–35 nights. Keep positive finite totals, EUR currency and successful availability in the raw quotation. Cost = total / nights × 30. This is a normalized quoted accommodation amount, not an ordinary residential rent or a guaranteed checkout total including all taxes/fees.


In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from src.study_utils import load_spain, export_table, save_figure, SPAIN
from src.cities import CITIES

data = load_spain()
plt.rcParams.update({'font.size': 10, 'axes.spines.top': False, 'axes.spines.right': False})
from src.study_utils import quote_metadata
import seaborn as sns


In [2]:
meta = data.price_quote_raw.map(quote_metadata)
data['quote_currency'] = meta.map(lambda x: x[0])
data['quote_available'] = meta.map(lambda x: x[1])
room_ok = data.room_type.isin(['Entire home/apt', 'Private room'])
duration_ok = data.quote_nights.between(28, 35)
amount_ok = np.isfinite(data.quote_total) & data.quote_total.gt(0)
metadata_ok = data.quote_currency.eq('EUR') & data.quote_available.eq(True)
valid = room_ok & duration_ok & amount_ok & metadata_ok
monthly = data.loc[valid].copy()
monthly['cost_30'] = monthly.quote_total / monthly.quote_nights * 30
assert np.isfinite(monthly.cost_30).all() and monthly.cost_30.gt(0).all()
audit = []
for city in SPAIN:
    c = data.city.eq(city)
    audit.append({'city':city, 'all_listings':int(c.sum()), 'target_room_types':int((c & room_ok).sum()),
                  'quote_28_35':int((c & room_ok & duration_ok).sum()),
                  'positive_total':int((c & room_ok & duration_ok & amount_ok).sum()),
                  'valid_quotes':int((c & valid).sum())})
display(export_table(pd.DataFrame(audit), 'study07_filter_audit'))
summary = monthly.groupby(['city_name', 'room_type']).cost_30.agg(
    n='size', q1=lambda s:s.quantile(.25), median='median', q3=lambda s:s.quantile(.75))
display(export_table(summary, 'study07_monthly_cost', index=True))


,city,all_listings,target_room_types,quote_28_35,positive_total,valid_quotes
0,barcelona,15293,15106,5076,4956,4956
1,madrid,22708,22510,2811,2767,2767


n          q1       median           q3
city_name room_type                                                  
Barcelona Entire home/apt  3329  2295.93750  3052.345161  4081.935484
          Private room     1627   970.91129  1156.875000  1549.284375
Madrid    Entire home/apt  2175  2325.50500  3023.003226  4194.385714
          Private room      592   784.89000   956.419355  1202.332258

In [3]:
dates = monthly.groupby(['city_name', 'room_type']).agg(
    n=('id', 'size'), checkin_min=('price_quote_checkin_date', 'min'),
    checkin_median=('price_quote_checkin_date', 'median'), checkin_max=('price_quote_checkin_date', 'max'),
    nights_min=('quote_nights', 'min'), nights_max=('quote_nights', 'max'))
display(export_table(dates, 'study07_quote_dates', index=True))
monthly['checkin_month'] = monthly.price_quote_checkin_date.dt.strftime('%Y-%m')
months = monthly.groupby(['city_name', 'room_type', 'checkin_month']).size().rename('n').reset_index()
export_table(months, 'study07_quote_months')
display(pd.crosstab(monthly.city_name, monthly.checkin_month))
fig, ax = plt.subplots(figsize=(9, 5))
sns.boxplot(data=monthly, x='city_name', y='cost_30', hue='room_type',
            order=['Barcelona','Madrid'], hue_order=['Entire home/apt','Private room'],
            showfliers=False, palette=['#235789', '#e09f3e'], ax=ax)
ax.set(xlabel='', ylabel='30-night equivalent quoted cost (EUR)', title='Stays of 28–35 nights: all valid quotes')
ax.legend(title='Accommodation type')
fig.tight_layout()
save_figure(fig, 'study07_monthly_cost')
plt.show()


n checkin_min checkin_median checkin_max  \
city_name room_type                                                      
Barcelona Entire home/apt  3329  2026-06-24     2026-08-06  2027-06-01   
          Private room     1627  2026-06-24     2026-07-25  2027-06-01   
Madrid    Entire home/apt  2175  2026-06-20     2026-07-25  2027-06-01   
          Private room      592  2026-06-20     2026-07-24  2027-06-01   

                           nights_min  nights_max  
city_name room_type                                
Barcelona Entire home/apt        28.0        35.0  
          Private room           28.0        35.0  
Madrid    Entire home/apt        28.0        35.0  
          Private room           28.0        35.0

checkin_month,2026-06,2026-07,2026-08,2026-09,2026-10,2026-11,2026-12,2027-01,2027-02,2027-03,2027-04,2027-05,2027-06
city_name,,,,,,,,,,,,,
Barcelona,924,1385,906,491,428,222,248,169,67,36,45,31,4
Madrid,597,994,411,164,153,87,122,151,32,18,11,19,8


/var/folders/6n/6h2dvkgn1x1bmhbn601c5xk80000gn/T/ipykernel_11856/1606129181.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Findings and limitations
Read each median with its sample size and interquartile range. All valid quotes contribute to the statistics and box construction; hiding outlier points does not remove them. The check-in summary and month table document differing seasonal coverage. Multiplying by 30/nights provides a comparison unit, not a new booking quote; discounts can be nonlinear. There is one recorded quote per listing, with no matched dates or guest counts. Missing quotes and unknown fee coverage limit representativeness. No inference about affordability for residents is made.


The entire-home medians are close: EUR 3,052 in Barcelona and EUR 3,023 in Madrid. Private-room medians are EUR 1,157 and EUR 956. There are 3,329 / 1,627 qualifying Barcelona entire-home / private-room quotes and 2,175 / 592 in Madrid. Check-ins extend from June 2026 to June 2027, with a later median date for Barcelona entire homes. No raw quote in this retained sample reports non-null tax, service-fee or cleaning-fee components; their inclusion in the total cannot be verified.